<a href="https://colab.research.google.com/github/teoteoh/sctec_aulas/blob/main/Miniprojeto_TeodoraCosta_Analise_de_Dados_TI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importando as bibliotecas relevantes:

In [1]:
import csv
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

Carregando CSV e fazendo as primeiras
inspeções:

In [2]:
df = pd.read_csv('Base Varejo.csv', sep=';')

display(df.head())


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534.0,M,4.0,1.0,C,67.0,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534.0,M,4.0,1.0,C,70.0,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534.0,M,4.0,1.0,C,178.0,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534.0,M,4.0,1.0,C,4.0,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534.0,M,4.0,1.0,C,175.0,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286636 entries, 0 to 286635
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         286636 non-null  object 
 1   CO_ID        286636 non-null  int64  
 2   CL_ID        286635 non-null  float64
 3   CL_GENERO    286635 non-null  object 
 4   CL_EC        286635 non-null  float64
 5   CL_FHL       286635 non-null  float64
 6   CL_SEG       286635 non-null  object 
 7   PR_ID        286635 non-null  float64
 8   PR_CAT       286635 non-null  object 
 9   PR_NOME      286635 non-null  object 
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(8), int64(1), object(5)
memory usage: 30.6+ MB
None


Há colunas sem rótulo e com valores vazios. Fazemos a remoção das colunas vazias para manter a base de dados mais limpa e concisa:

In [4]:
df = df.dropna(axis=1, how='all')

Checamos para ver se há linhas duplicadas:

In [6]:
total_duplicadas = df.duplicated().sum()


if total_duplicadas > 0:
    print(f"Linhas duplicadas: {total_duplicadas}")
else:
    print("\nNão há duplicatas.")

Linhas duplicadas: 33257


Agora removemos duplicatas e checamos o resultado:

In [7]:
df = df.drop_duplicates()

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 253379 entries, 0 to 286635
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   DATA       253379 non-null  object 
 1   CO_ID      253379 non-null  int64  
 2   CL_ID      253378 non-null  float64
 3   CL_GENERO  253378 non-null  object 
 4   CL_EC      253378 non-null  float64
 5   CL_FHL     253378 non-null  float64
 6   CL_SEG     253378 non-null  object 
 7   PR_ID      253378 non-null  float64
 8   PR_CAT     253378 non-null  object 
 9   PR_NOME    253378 non-null  object 
dtypes: float64(4), int64(1), object(5)
memory usage: 21.3+ MB
None


Transformando a coluna DATA em DataTime:

In [8]:
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y')

display(df.head())
print(df['DATA'].dtype)

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,2019-02-01,1000,534.0,M,4.0,1.0,C,67.0,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534.0,M,4.0,1.0,C,70.0,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534.0,M,4.0,1.0,C,178.0,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534.0,M,4.0,1.0,C,4.0,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534.0,M,4.0,1.0,C,175.0,LIMPEZA,LIMPADOR MULTIUSO


datetime64[ns]


Agora verificamos se há categorias vazias:

In [11]:
print("NaN por coluna:")
print(df.isna().sum())

print("Strings por coluna:")
for col in df.select_dtypes(include=['object']).columns:
    vazios = (df[col].astype(str).str.strip() == '').sum()
    if vazios > 0:
        print(f"{col}: {vazios} registros vazios.")
    else:
        print(f"{col}: Nenhum registro vazio.")

NaN por coluna:
DATA         0
CO_ID        0
CL_ID        1
CL_GENERO    1
CL_EC        1
CL_FHL       1
CL_SEG       1
PR_ID        1
PR_CAT       1
PR_NOME      1
dtype: int64
Strings por coluna:
CL_GENERO: Nenhum registro vazio.
CL_SEG: Nenhum registro vazio.
PR_CAT: Nenhum registro vazio.
PR_NOME: Nenhum registro vazio.


Substituimos os NaN por "Sem Categoria" e checamos o resultado:

In [13]:
df = df.fillna('Sem Categoria')

print(f"Valores vazios: {df.isna().sum().sum()}")

display(df.tail())

Valores vazios: 0


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
286629,2020-10-05,317752,538.0,M,3.0,0.0,B,142.0,BEBIDAS,REFRIGERANTE LARANJA
286631,2020-10-05,317752,538.0,M,3.0,0.0,B,57.0,ALIMENTOS,REQUEIJAO
286632,2020-10-05,317752,538.0,M,3.0,0.0,B,186.0,HIGIENE,HIDRATANTE
286633,2020-10-05,317752,538.0,M,3.0,0.0,B,172.0,LIMPEZA,LUSTRA MOVEIS
286635,2020-10-05,317,Sem Categoria,Sem Categoria,Sem Categoria,Sem Categoria,Sem Categoria,Sem Categoria,Sem Categoria,Sem Categoria


Estatísticas básicas relacionadas ao número de filhos dos clientes:

In [10]:
stats = df['CL_FHL'].describe()
mode_val = df['CL_FHL'].mode()[0]
median_val = df['CL_FHL'].median()

print("Estatísticas Descritivas Referentes ao Número de Filhos:")
print(f"Contagem:        {stats['count']}")
print(f"Média:           {stats['mean']:.2f}")
print(f"Mediana:         {median_val}")
print(f"Moda:            {mode_val}")
print(f"Desvio Padrão:   {stats['std']:.2f}")
print(f"Mínimo:          {stats['min']}")
print(f"Máximo:          {stats['max']}")
print(f"Quartil 25%:     {stats['25%']}")
print(f"Quartil 50%:     {stats['50%']}")
print(f"Quartil 75%:     {stats['75%']}")

Estatísticas Descritivas Referentes ao Número de Filhos:
Contagem:        253378.0
Média:           1.14
Mediana:         0.0
Moda:            0.0
Desvio Padrão:   1.42
Mínimo:          0.0
Máximo:          4.0
Quartil 25%:     0.0
Quartil 50%:     0.0
Quartil 75%:     2.0
